# Tasks 3–8 Pipeline (Preprocess → Features → BiLSTM Multi-head → Evaluation → Personalization)

This notebook executes tasks 3–8 using the real data in this repository.

Implemented components:
- Task 3: keypoint preprocessing (imputation, normalization, FPS synchronization)
- Task 4: biomechanical angle feature computation
- Task 5/6: BiLSTM multi-head model (exercise classification + quality regression)
- Task 7: metrics + ablation (raw coordinates vs angles)
- Task 8: personalization adapter fine-tuning


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from fitness_adapt.dataset_builder import DatasetConfig, build_split_windows
from fitness_adapt.io_utils import load_json
from fitness_adapt.project import ProjectPaths
from fitness_adapt.train_eval import TrainConfig, personalize_on_user_windows, run_ablation, train_multitask_bilstm

import numpy as np
import torch

paths_tmp = ProjectPaths.from_root(PROJECT_ROOT)
PROJECT_ROOT = paths_tmp.root
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))


In [ ]:
paths = ProjectPaths.from_root(PROJECT_ROOT)
paths.processed_dir.mkdir(parents=True, exist_ok=True)
paths.outputs_dir.mkdir(parents=True, exist_ok=True)

# Use subsets for iteration speed in notebook; switch limits for fuller runs.
train_keys = load_json(paths.split_path('train'))[:20]
val_keys = load_json(paths.split_path('val'))[:8]

dcfg = DatasetConfig(
    target_fps=30.0,
    window_size=30,
    window_stride=15,
    frame_stride=6,
    max_frames_per_video=120,
    conf_threshold=0.35,
    yolo_weights='yolo11n-pose.pt',
    device='cpu',
)

train_data = build_split_windows('train', paths=paths, config=dcfg, keys_override=train_keys)
val_data = build_split_windows('val', paths=paths, config=dcfg, keys_override=val_keys)

print('train windows', train_data['x_angles'].shape, train_data['x_raw'].shape)
print('val windows', val_data['x_angles'].shape, val_data['x_raw'].shape)


In [ ]:
tcfg = TrainConfig(batch_size=32, lr=1e-3, epochs=6, device='cpu')
num_classes = int(np.max(train_data['y_exercise'])) + 1

ablation = run_ablation(train_data, val_data, cfg=tcfg, num_classes=num_classes)
print('ablation', ablation)

model, val_metrics = train_multitask_bilstm(
    train_data['x_angles'],
    train_data['y_exercise'],
    train_data['y_quality'],
    val_data['x_angles'],
    val_data['y_exercise'],
    val_data['y_quality'],
    input_dim=train_data['x_angles'].shape[-1],
    num_classes=num_classes,
    cfg=tcfg,
)
print('val_metrics_angle_model', val_metrics)

torch.save(model.state_dict(), paths.outputs_dir / 'bilstm_multitask.pt')


In [ ]:
key_ids = train_data['key_ids']
first_user = key_ids[0]
mask = key_ids == first_user
user_metrics = personalize_on_user_windows(
    model,
    train_data['x_angles'][mask],
    train_data['y_exercise'][mask],
    train_data['y_quality'][mask],
    cfg=tcfg,
    adapter_steps=20,
)
print('user key', first_user)
print('personalization metrics', user_metrics)


## Task 9

A runnable real-time integration script is available:

- `python3 scripts/run_task9_realtime_demo.py --video /workspace/videos_squat/32903_8.mp4`

It shows:
- live skeleton overlay,
- rolling-window exercise classification,
- quality score + actionable text feedback.
